In [60]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from joblib import dump
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, matthews_corrcoef, confusion_matrix

In [61]:
param_grid = {
    "n_estimators": [100, 500, 1000], 
    "criterion": ["gini", "entropy"],           
    "min_samples_split": [2, 5, 10],           
    "min_samples_leaf": [1, 2, 4],              
    "max_features": ["auto", "sqrt", "log2"],  
    "max_depth": [10, 20, 30]
    }

In [62]:
def undersampling(df_data, seed):
    ten_percent=df_data[df_data["target"]==2].sample(frac=0.10, random_state=42)
    X=df_data.drop('target', axis=1)
    y=df_data['target']  
    #Se definen los objetos para submuestrear
    X['index']=X.index 
    undersampler=RandomUnderSampler(sampling_strategy='not minority', random_state=42)    

    #Se aplica el submuestreo
    X_res, y_res=undersampler.fit_resample(X, y)
    df_resampled=pd.concat([X_res,y_res], axis=1)

    index_res=X_res['index']
    df_resampled.drop('index', axis=1, inplace=True)
    display(df_resampled)
    mask=~X['index'].isin(index_res)
    excluded_data=df_data[mask.values]
    data_independent= pd.concat([ten_percent, excluded_data], axis=0)
    data_independent.reset_index(drop=True, inplace=True)
    data_independent.to_csv("../../models/data/data_independent.csv", index=False)
    
    return df_resampled

In [63]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [64]:
def split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [65]:
def metrics(predict_val, y_val, dataset, div):
    acc_value = accuracy_score(y_pred=predict_val, y_true=y_val) 
    recall_value = recall_score(y_pred=predict_val, y_true=y_val, average='weighted')
    precision_value = precision_score(y_pred=predict_val, y_true=y_val, average='weighted') 
    f1_value = f1_score(y_pred=predict_val, y_true=y_val, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict_val, y_true=y_val)
    cm = confusion_matrix(y_pred=predict_val, y_true=y_val)
    cm_df = pd.DataFrame(cm)
    cm_dict = cm_df.to_dict()

    df_metrics = pd.DataFrame([[dataset, "Random_Forest", div, acc_value, recall_value, precision_value, f1_value, mcc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "conf_matrix"])

    return df_metrics

In [66]:
def train(train, val, div, seed):
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"Train Random Forest with seed {seed} and division {div}")
    results = []
    rf = RandomForestClassifier(random_state=seed)
    rf.fit(X_train, y_train)

    dump(rf, f"../../models/RandomForest_Grid_{seed}_{div}.joblib")

    y_pred_train = rf.predict(X_train)
    y_pred_val = rf.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Grid_{seed}_{div}_predictions.csv", index=False)
    
    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results.append(train_metrics)
    results.append(val_metrics)
    return pd.concat(results, ignore_index=True), rf

In [67]:
def grid_function(rf, train, val, seed, div): 
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"GridSearchCV for Random Forest with seed {seed} and division {div}")
    grid = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_

    print(f"Best estimators: {best_model}")
    print(f"Best parameters found: {best_params}")
    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/RandomForest_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)
    dump(best_model, f"../../models/RandomForest_Grid_{seed}_{div}_best.joblib")

    y_pred_train = best_model.predict(X_train)
    y_pred_val = best_model.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Grid_{seed}_{div}_best_predictions.csv", index=False)

    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    return results

In [68]:
def main_train(df_data, seed, grid_search=False):
    all_metrics = []
    all_metrics_grid = []
    df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = split(df_data, seed)
    metrics_orig, model_orig = train(df_train, df_val, "Original", seed)
    metrics_under, model_under = train(df_train_under, df_val_under, "Under", seed)
    #metrics_over, model_over = train(df_train_over, df_val_over, "Over", seed)

    all_metrics = pd.concat([metrics_orig, metrics_under], ignore_index=True)
    all_metrics.to_csv(f"../../metrics/RandomForest_Grid_{seed}_metrics.csv", index=False)
    if grid_search:
        all_metrics_grid = pd.concat([grid_function(model_orig, df_train, df_val, seed, "Original"),
                                      grid_function(model_under, df_train_under, df_val_under, seed, "Under"),], 
                                      ignore_index=True)
                                      #grid_function(model_over, df_train_over, df_val_over, seed, "Over")
        all_metrics_grid.to_csv(f"../../metrics/RandomForest_Grid_{seed}_grid_metrics.csv", index=False)
    

In [69]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics", "half_life_seconds", "length_sequence"], axis=1, inplace=True)

In [70]:
df_data

,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10,...,p_1016,p_1017,p_1018,p_1019,p_1020,p_1021,p_1022,p_1023,p_1024,target
0,0.024928,0.038137,0.010136,0.013962,0.003674,-0.054007,0.023122,0.204115,-0.129186,-0.162114,...,-0.120455,0.058737,0.107394,0.060945,0.010294,0.168054,-0.020148,0.067316,-0.133364,1
1,-0.032362,-0.000383,-0.048013,0.011207,-0.005493,-0.058146,-0.051805,-0.028580,0.028206,-0.030951,...,-0.084771,0.062330,-0.006251,-0.037678,0.033790,0.201147,0.025814,0.094777,-0.116653,1
2,0.056347,-0.045785,-0.013331,-0.050032,-0.088194,-0.073964,-0.068582,0.031079,-0.001762,-0.124785,...,-0.031063,0.087633,0.120435,0.018659,0.031999,0.077117,-0.017236,0.066621,-0.089538,1
3,-0.011724,0.112869,-0.010287,-0.013133,-0.023317,-0.039255,-0.106493,0.041725,-0.067948,0.025545,...,0.070505,0.051101,0.074827,0.041634,0.000973,0.049307,0.021370,0.047843,-0.033902,1
4,-0.105726,0.069741,-0.127632,-0.124845,0.202506,0.030408,-0.049978,0.130994,0.033893,0.089469,...,-0.020812,-0.006855,0.066739,-0.082009,-0.066162,0.287695,0.050576,0.212813,0.008588,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,-0.100075,0.001455,0.017908,0.044346,-0.079563,0.011130,-0.021392,0.009696,0.048142,0.042567,...,-0.104691,0.052998,0.100753,-0.029305,0.093444,0.089840,0.031600,0.066225,-0.084514,1
1309,-0.117940,0.047649,-0.007363,-0.014420,0.019296,0.058130,-0.071593,0.036150,0.075177,-0.137731,...,-0.056583,0.050460,0.018345,0.036152,-0.014890,0.138495,-0.086376,0.005198,-0.064887,0
1310,-0.203825,0.062788,0.006884,-0.049276,0.067048,0.083548,-0.154984,0.052247,0.098920,-0.150795,...,-0.048853,-0.012628,0.023684,0.065029,0.002418,0.121806,-0.107974,-0.035496,-0.073660,1
1311,-0.203825,0.062788,0.006884,-0.049276,0.067048,0.083548,-0.154984,0.052247,0.098920,-0.150795,...,-0.048853,-0.012628,0.023684,0.065029,0.002418,0.121806,-0.107974,-0.035496,-0.073660,2


In [71]:
folder = "../../data/numerical_rep/"
seed= 42

In [ ]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_{repr_name}.csv"
main_train(df_data, seed, grid_search=True)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing ProtT5


,p_1,p_2,p_3,p_4,p_5,p_6,p_7,p_8,p_9,p_10,...,p_1016,p_1017,p_1018,p_1019,p_1020,p_1021,p_1022,p_1023,p_1024,target
1218,-0.168331,0.019908,-0.165182,-0.077062,-0.011016,0.004051,-0.120691,0.054929,0.120765,0.127900,...,-0.048451,0.174067,0.015163,0.001412,-0.041287,0.106463,-0.096717,0.130232,-0.055155,0
634,-0.000510,0.031792,-0.118280,-0.027885,0.026655,-0.054459,-0.154253,0.105561,0.051154,0.026705,...,-0.012568,0.003288,0.001773,0.012548,0.061004,0.082979,-0.033267,0.019713,-0.232028,0
160,-0.284822,-0.104412,-0.088315,0.024645,-0.082899,0.008999,0.043135,0.153930,-0.012589,-0.046071,...,-0.047826,0.142586,-0.055921,0.064731,-0.073670,0.073301,-0.067256,0.026223,-0.097176,0
726,0.034958,-0.091921,-0.062912,0.028036,0.000790,-0.055446,-0.133578,0.096255,-0.023717,-0.013008,...,-0.086651,0.104678,0.038423,0.003866,0.082815,0.195643,-0.127895,-0.017944,-0.070040,0
215,0.008061,-0.026131,-0.127803,-0.039956,-0.130736,0.005296,0.058935,0.098008,-0.072140,0.039576,...,-0.099672,0.047897,-0.113072,0.032443,-0.039043,-0.048026,0.017255,0.151638,-0.050290,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1299,-0.137362,-0.114130,-0.120217,-0.007904,0.254243,-0.015313,-0.092671,0.073956,0.105611,-0.084017,...,-0.135456,0.003368,-0.034771,0.066850,-0.076335,0.196156,-0.066858,0.017583,-0.065645,2
1302,0.040809,0.098313,-0.147429,0.019754,0.005857,-0.113292,-0.058609,-0.027246,0.054182,0.107246,...,-0.068945,0.237313,-0.024556,0.134004,0.052650,0.192038,-0.219479,0.126286,0.024984,2
1304,-0.098615,-0.091629,-0.020671,-0.007833,0.059455,0.046928,-0.202553,0.034373,0.059682,0.020352,...,-0.060711,0.032902,0.001195,-0.016447,0.030442,0.086734,-0.125661,0.110023,-0.169574,2
1307,-0.100075,0.001455,0.017908,0.044346,-0.079563,0.011130,-0.021392,0.009696,0.048142,0.042567,...,-0.104691,0.052998,0.100753,-0.029305,0.093444,0.089840,0.031600,0.066225,-0.084514,2


Train Random Forest with seed 42 and division Original
Train Random Forest with seed 42 and division Under
GridSearchCV for Random Forest with seed 42 and division Original
